# 06 – Bronze: full load from the Dog CEO API into an Iceberg table + schema evolution

**Goal (Task 3, parts 2 and 3):**

1. Upgrade the Task 2 **full** API load (`replace`) so it writes an **Iceberg table** `bronze.dogceo_breeds`, partitioned by ingestion date.
2. Read it with **PyIceberg** and verify.
3. **Schema evolution:** add a new column, rerun, and show that the schema changed but the **existing data files were not rewritten**.

In [ ]:
import os
import dlt
import pandas as pd
from dlt.sources.helpers import requests
from lake import PIPELINES_DIR, add_metadata
from pile import (BRONZE, get_catalog, pile_destination, partition_by_ingestion_date, run_to_pile,
                  list_pile, show_catalog_db, snapshots, data_files)

DOG_API_URL = os.getenv("DOG_API_URL", "https://dog.ceo/api/breeds/list/all")
pd.set_option("display.max_colwidth", 120)
catalog = get_catalog("silo")

## Step 1 – The resource

Same as notebook 03, with one switch: `new_column=...` adds one **extra column** to every row (used in step 4 for schema evolution).

`NEW_COLUMNS` has two choices. A column can be added only once, so if you already showed `is_sub_breed`, use `name_length` the next time.

In [ ]:
NEW_COLUMNS = {
    "is_sub_breed": lambda breed, sub: sub is not None,                   # True for "australian shepherd"
    "name_length":  lambda breed, sub: len(f"{breed} {sub}" if sub else breed),
}


def build_api_resource(new_column: str = None):

    @dlt.resource(name="dogceo_breeds")
    def dog_breeds():
        response = requests.get(DOG_API_URL, timeout=30)
        response.raise_for_status()
        for breed, sub_breeds in response.json()["message"].items():
            for sub in (sub_breeds or [None]):          # [None] = breed without sub-breeds
                row = {
                    "breed": breed,
                    "sub_breed": sub,
                    "full_name": f"{breed} {sub}" if sub else breed,
                    "images_api": f"https://dog.ceo/api/breed/{breed}/{sub}/images" if sub
                                  else f"https://dog.ceo/api/breed/{breed}/images",
                }
                if new_column:                         # <- the NEW column (schema evolution)
                    row[new_column] = NEW_COLUMNS[new_column](breed, sub)
                yield row

    resource = dog_breeds()
    add_metadata(resource, source_system="dogceo", source_resource="breeds/list/all", ingestion_type="full")
    return partition_by_ingestion_date(resource)


api_pipeline = dlt.pipeline(
    pipeline_name="pile_api",
    destination=pile_destination("silo"),
    dataset_name=BRONZE,
    pipelines_dir=PIPELINES_DIR,
)

## Step 2 – Full load (`replace`) into Iceberg

In [ ]:
run_to_pile(api_pipeline, build_api_resource(), write_disposition="replace")
print("Tables in bronze:", catalog.list_tables(BRONZE))
list_pile("bronze/dogceo_breeds")

## Step 3 – Read with PyIceberg and verify

Compare with a direct call to the API: same number of breeds, no duplicates, metadata filled.

In [ ]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
breeds = breeds_table.scan().to_pandas()

api_data = requests.get(DOG_API_URL, timeout=30).json()["message"]
expected = sum(max(len(subs), 1) for subs in api_data.values())

print("Rows in Iceberg table:", len(breeds))
print("Rows expected from API:", expected, "→ OK" if len(breeds) == expected else "→ MISMATCH")
print("Duplicate full_name:  ", breeds.full_name.duplicated().sum())
print("Partition:", breeds_table.spec())
breeds.head()

## Step 4 – Schema evolution

Save the **schema** and the **data files** BEFORE the change:

In [ ]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
schema_before = breeds_table.schema()
snapshot_before = breeds_table.current_snapshot().snapshot_id
files_before = data_files(breeds_table)
objects_before = list_pile("bronze/dogceo_breeds/data")

print("Schema id BEFORE:", schema_before.schema_id)
print(schema_before)
files_before

Now rerun the import **with the new column** `is_sub_breed` (or `name_length` if `is_sub_breed` already exists). dlt sees the new column and asks Iceberg to **add it to the schema** (`update_schema().union_by_name(...)`).

In [ ]:
run_to_pile(api_pipeline, build_api_resource(new_column="is_sub_breed"), write_disposition="replace")

In [ ]:
breeds_table = catalog.load_table(f"{BRONZE}.dogceo_breeds")
schema_after = breeds_table.schema()
print("Schema id AFTER:", schema_after.schema_id)
print(schema_after)

In [ ]:
# Compare the two schemas
before_cols = {f.name for f in schema_before.fields}
after_cols = {f.name for f in schema_after.fields}
print("Added columns:  ", after_cols - before_cols)
print("Removed columns:", before_cols - after_cols)
print("\nAll schema versions Iceberg remembers:")
for schema_id, schema in breeds_table.schemas().items():
    print(f"  schema {schema_id}: {len(schema.fields)} columns")

### The existing data files were NOT rewritten

- The old Parquet file is still in the bucket with the **same key, size and last-modified time**.
- The old snapshot still points to it and is read with the **old schema** (without the new column).
- Adding a column only wrote **new metadata**. The new rows went into a **new** data file.

In [ ]:
objects_after = list_pile("bronze/dogceo_breeds/data")
compare = objects_before.merge(objects_after, on="key", how="outer", suffixes=("_before", "_after"), indicator=True)
compare["_merge"] = compare["_merge"].map({"both": "unchanged (still there)", "right_only": "NEW file",
                                          "left_only": "removed"})
compare

In [ ]:
snapshots(breeds_table)      # replace = 'delete' + 'append' snapshot; schema_id goes 0 → 1

In [ ]:
print("Old snapshot → its data files:")
display(data_files(breeds_table, snapshot_before))
print("Old snapshot read (old schema):")
display(breeds_table.scan(snapshot_id=snapshot_before).to_pandas().head(3))
print("Current table (new schema):")
breeds_table.scan().to_pandas()[["full_name", "sub_breed", *sorted(after_cols - before_cols)]].head(5)